In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import pandas as pd

grocery_dataset = pd.read_csv('/content/drive/MyDrive/Data Mining Lab/Groceries_dataset.csv')
print(grocery_dataset.info())

grocery_dataset.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38765 entries, 0 to 38764
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Member_number    38765 non-null  int64 
 1   Date             38765 non-null  object
 2   itemDescription  38765 non-null  object
dtypes: int64(1), object(2)
memory usage: 908.7+ KB
None


,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [2]:
# group the transaction as Member_number+Date

raw_transactions = grocery_dataset.groupby(['Member_number','Date'])['itemDescription'].apply(list).values.tolist()

print('Transactions (grouped by member_number & Date): ')
print(raw_transactions)

Transactions (grouped by member_number & Date): 
[['sausage', 'whole milk', 'semi-finished bread', 'yogurt'], ['whole milk', 'pastry', 'salty snack'], ['canned beer', 'misc. beverages'], ['sausage', 'hygiene articles'], ['soda', 'pickled vegetables'], ['frankfurter', 'curd'], ['sausage', 'whole milk', 'rolls/buns'], ['whole milk', 'soda'], ['beef', 'white bread'], ['frankfurter', 'soda', 'whipped/sour cream'], ['frozen vegetables', 'other vegetables'], ['butter', 'whole milk'], ['tropical fruit', 'sugar'], ['butter milk', 'specialty chocolate'], ['sausage', 'rolls/buns'], ['root vegetables', 'detergent'], ['frozen meals', 'dental care'], ['rolls/buns', 'rolls/buns'], ['dish cleaner', 'cling film/bags'], ['canned beer', 'frozen fish'], ['other vegetables', 'hygiene articles'], ['pip fruit', 'whole milk', 'tropical fruit'], ['rolls/buns', 'red/blush wine', 'chocolate'], ['other vegetables', 'shopping bags'], ['whole milk', 'chocolate', 'packaged fruit/vegetables', 'rolls/buns'], ['root v

In [3]:
# deduplicate the items in the transactions
# since Apriori and FP-growth requires items in the transaction to be unique

transactions = []
for transaction in raw_transactions:
    transactions.append(list(set(transaction)))

print('Transactions(deduplicated): ')
print(transactions)

Transactions(deduplicated): 
[['yogurt', 'sausage', 'whole milk', 'semi-finished bread'], ['whole milk', 'salty snack', 'pastry'], ['misc. beverages', 'canned beer'], ['hygiene articles', 'sausage'], ['pickled vegetables', 'soda'], ['curd', 'frankfurter'], ['rolls/buns', 'sausage', 'whole milk'], ['soda', 'whole milk'], ['white bread', 'beef'], ['whipped/sour cream', 'soda', 'frankfurter'], ['frozen vegetables', 'other vegetables'], ['whole milk', 'butter'], ['tropical fruit', 'sugar'], ['butter milk', 'specialty chocolate'], ['rolls/buns', 'sausage'], ['detergent', 'root vegetables'], ['frozen meals', 'dental care'], ['rolls/buns'], ['cling film/bags', 'dish cleaner'], ['frozen fish', 'canned beer'], ['hygiene articles', 'other vegetables'], ['pip fruit', 'tropical fruit', 'whole milk'], ['rolls/buns', 'chocolate', 'red/blush wine'], ['other vegetables', 'shopping bags'], ['packaged fruit/vegetables', 'rolls/buns', 'whole milk', 'chocolate'], ['whole milk', 'root vegetables', 'pastry'

In [4]:
no_transactions = len(transactions)

print('No of transactions: ', no_transactions)

No of transactions:  14963


# Apriori

In [5]:
min_support_freq = 50

# Apriori algorithm
items = grocery_dataset['itemDescription'].unique()
print('No of items: ', len(items))

# # 1-itemset
# freq_itemset = {}
# for item in items:
#   count = 0
#   for transaction in transactions:
#     if item in transaction: count += 1

#   if count >= min_support_freq:
#     freq_itemset[item] = count

# print(freq_itemset)

No of items:  167


In [6]:
from itertools import combinations

def generate_candidates(prev_freq_sets, k):
  items = set()
  for itemset in prev_freq_sets:
    items.update(itemset if isinstance(itemset, tuple) else (itemset,))
  return list(combinations(items, k))

def get_support_frequency(itemsets, transactions, min_support_freq):
  freq_sets = {}
  for itemset in itemsets:
    count = 0
    for transaction in transactions:
      if all(item in transaction for item in itemset):
        count += 1
    if count >= min_support_freq:
      freq_sets[itemset] = count
  return freq_sets


freq_itemsets = []
L1 = {}
for item in items:
  count = sum(1 for t in transactions if item in t)
  if count >= min_support_freq:
    L1[(item,)] = count

freq_itemsets.append(L1)

k = 2
if not freq_itemsets[-1]:
  print("No frequent 1-itemsets found.")
else:
  while True:
    prev_L = freq_itemsets[-1]
    candidates = generate_candidates(prev_L.keys(), k)
    Lk = get_support_frequency(candidates, transactions, min_support_freq)
    if not Lk:
      break
    freq_itemsets.append(Lk)
    # print(f"Frequent {k}-itemsets:", Lk)
    k += 1

In [ ]:
print(len(freq_itemsets[0]), len(freq_itemsets[1]))
print(len(freq_itemsets))

105 90
2


In [ ]:
print(freq_itemsets)

[{('tropical fruit',): 1014, ('whole milk',): 2363, ('pip fruit',): 734, ('other vegetables',): 1827, ('rolls/buns',): 1646, ('pot plants',): 117, ('citrus fruit',): 795, ('beef',): 508, ('frankfurter',): 565, ('chicken',): 417, ('butter',): 527, ('fruit/vegetable juice',): 509, ('packaged fruit/vegetables',): 127, ('chocolate',): 353, ('specialty bar',): 209, ('butter milk',): 263, ('bottled water',): 908, ('yogurt',): 1285, ('sausage',): 903, ('brown bread',): 563, ('hamburger meat',): 327, ('root vegetables',): 1041, ('pork',): 555, ('pastry',): 774, ('canned beer',): 702, ('berries',): 326, ('coffee',): 473, ('misc. beverages',): 236, ('ham',): 256, ('turkey',): 80, ('red/blush wine',): 157, ('frozen potato products',): 72, ('flour',): 146, ('sugar',): 265, ('frozen meals',): 251, ('herbs',): 158, ('soda',): 1453, ('detergent',): 129, ('grapes',): 216, ('processed cheese',): 152, ('newspapers',): 582, ('curd',): 504, ('pasta',): 121, ('finished products',): 64, ('beverages',): 248,

In [ ]:
frequency = {}
frequent_itemsets = []

for itemset in freq_itemsets:
    frequency.update(itemset)
    frequent_itemsets.extend(list(itemset.keys()))

print(len(frequency))
print(frequency)
print(len(frequent_itemsets))
print(frequent_itemsets)

195
{('tropical fruit',): 1014, ('whole milk',): 2363, ('pip fruit',): 734, ('other vegetables',): 1827, ('rolls/buns',): 1646, ('pot plants',): 117, ('citrus fruit',): 795, ('beef',): 508, ('frankfurter',): 565, ('chicken',): 417, ('butter',): 527, ('fruit/vegetable juice',): 509, ('packaged fruit/vegetables',): 127, ('chocolate',): 353, ('specialty bar',): 209, ('butter milk',): 263, ('bottled water',): 908, ('yogurt',): 1285, ('sausage',): 903, ('brown bread',): 563, ('hamburger meat',): 327, ('root vegetables',): 1041, ('pork',): 555, ('pastry',): 774, ('canned beer',): 702, ('berries',): 326, ('coffee',): 473, ('misc. beverages',): 236, ('ham',): 256, ('turkey',): 80, ('red/blush wine',): 157, ('frozen potato products',): 72, ('flour',): 146, ('sugar',): 265, ('frozen meals',): 251, ('herbs',): 158, ('soda',): 1453, ('detergent',): 129, ('grapes',): 216, ('processed cheese',): 152, ('newspapers',): 582, ('curd',): 504, ('pasta',): 121, ('finished products',): 64, ('beverages',): 2

In [ ]:
print('\nFrequent Itemsets:')
for itemset in frequent_itemsets:
    print(itemset)


Frequent Itemsets:
('tropical fruit',)
('whole milk',)
('pip fruit',)
('other vegetables',)
('rolls/buns',)
('pot plants',)
('citrus fruit',)
('beef',)
('frankfurter',)
('chicken',)
('butter',)
('fruit/vegetable juice',)
('packaged fruit/vegetables',)
('chocolate',)
('specialty bar',)
('butter milk',)
('bottled water',)
('yogurt',)
('sausage',)
('brown bread',)
('hamburger meat',)
('root vegetables',)
('pork',)
('pastry',)
('canned beer',)
('berries',)
('coffee',)
('misc. beverages',)
('ham',)
('turkey',)
('red/blush wine',)
('frozen potato products',)
('flour',)
('sugar',)
('frozen meals',)
('herbs',)
('soda',)
('detergent',)
('grapes',)
('processed cheese',)
('newspapers',)
('curd',)
('pasta',)
('finished products',)
('beverages',)
('bottled beer',)
('dessert',)
('dog food',)
('specialty chocolate',)
('condensed milk',)
('white wine',)
('meat',)
('ice cream',)
('hard cheese',)
('cream cheese ',)
('liquor',)
('pickled vegetables',)
('liquor (appetizer)',)
('UHT-milk',)
('candy',)
('o

In [ ]:
def get_support(item):
  return frequency.get(item, 0) / no_transactions   # return 0 if item is not found

def get_confidence(antecedent, consequent):
  union_itemset = tuple(sorted(antecedent + consequent))
  antecedent_support = get_support(antecedent)
  if antecedent_support == 0:
    return 0
  return get_support(union_itemset) / antecedent_support

print(get_support(('tropical fruit',)))
print(get_confidence(('tropical fruit',),('root vegetables',)))
print(get_confidence(('root vegetables',),('tropical fruit',)))

0.0677671589921807
0.0
0.0


In [ ]:
from itertools import combinations

minimum_confidence = 0.7

association_rules = []

for grouped_itemset in frequent_itemsets:
  if len(grouped_itemset[0]) < 2:
   continue

  for itemset in grouped_itemset:    # here itemset is an array of 1,2,..n-itemset
    for i in range(1, len(itemset)):
      for antecedent in combinations(itemset, i):
        consequent = tuple(set(itemset) - set(antecedent))
        if antecedent in frequency and consequent in frequency:
          confidence = get_confidence(antecedent, consequent)
          if confidence >= minimum_confidence:
            association_rules.append(f"[{antecedent} => {consequent}]")

print("Association rules:")
for rule in association_rules: print(rule)

In [ ]:
minimum_confidence = 0.7

def get_association_rules(frequent_itemsets):
    association_rules = []

    for itemset, _ in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):
                consequent = tuple(set(itemset) - set(antecedent))
                if antecedent in frequency and consequent in frequency:
                    confidence = get_confidence(antecedent, consequent)
                    if confidence >= minimum_confidence:
                        association_rules.append(f"[{antecedent} => {consequent}]")
    return association_rules


association_rules = get_association_rules(frequency)
print("Association rules:")
for rule in association_rules:
   print(rule)

print(f"\nTotal association rules generated: {len(association_rules)}")

Association rules:

Total association rules generated: 0


# FP-growth

In [ ]:
def get_item_frequencies(transactions):
  frequency_table = {}
  for transaction in transactions:
    for item in transaction:
      if item not in frequency_table:
        frequency_table[item] = 1
      else:
        frequency_table[item] += 1
  return frequency_table


frequency_table = get_item_frequencies(transactions)
# print(frequency_table)

# drop the items less than min support freq
min_support_freq = 50
frequency_table = {item: count for item, count in frequency_table.items() if count >= min_support_freq}
print(frequency_table)

{'sausage': 903, 'yogurt': 1285, 'semi-finished bread': 142, 'whole milk': 2363, 'salty snack': 281, 'pastry': 774, 'canned beer': 702, 'misc. beverages': 236, 'hygiene articles': 205, 'pickled vegetables': 134, 'soda': 1453, 'frankfurter': 565, 'curd': 504, 'rolls/buns': 1646, 'white bread': 359, 'beef': 508, 'whipped/sour cream': 654, 'frozen vegetables': 419, 'other vegetables': 1827, 'butter': 527, 'tropical fruit': 1014, 'sugar': 265, 'specialty chocolate': 239, 'butter milk': 263, 'root vegetables': 1041, 'detergent': 129, 'frozen meals': 251, 'cling film/bags': 74, 'dish cleaner': 73, 'frozen fish': 102, 'pip fruit': 734, 'chocolate': 353, 'red/blush wine': 157, 'shopping bags': 712, 'packaged fruit/vegetables': 127, 'margarine': 482, 'bottled water': 908, 'flour': 146, 'bottled beer': 678, 'chicken': 417, 'liquor (appetizer)': 67, 'dessert': 353, 'hamburger meat': 327, 'liver loaf': 50, 'white wine': 175, 'domestic eggs': 555, 'photo/film': 79, 'herbs': 158, 'newspapers': 582, 

In [ ]:
# order the transactions based on frequency table

def reorder_transactions(transactions, frequency_table):
  ordered_transactions = []
  for transaction in transactions:
    filtered_transaction = [item for item in transaction if item in frequency_table]
    filtered_transaction.sort(key=lambda item: frequency_table[item], reverse=True)
    if filtered_transaction:
      ordered_transactions.append(filtered_transaction)
  return ordered_transactions

ordered_transactions = reorder_transactions(transactions, frequency_table)
print(ordered_transactions)

[['whole milk', 'yogurt', 'sausage', 'semi-finished bread'], ['whole milk', 'pastry', 'salty snack'], ['canned beer', 'misc. beverages'], ['sausage', 'hygiene articles'], ['soda', 'pickled vegetables'], ['frankfurter', 'curd'], ['whole milk', 'rolls/buns', 'sausage'], ['whole milk', 'soda'], ['beef', 'white bread'], ['soda', 'whipped/sour cream', 'frankfurter'], ['other vegetables', 'frozen vegetables'], ['whole milk', 'butter'], ['tropical fruit', 'sugar'], ['butter milk', 'specialty chocolate'], ['rolls/buns', 'sausage'], ['root vegetables', 'detergent'], ['frozen meals'], ['rolls/buns'], ['cling film/bags', 'dish cleaner'], ['canned beer', 'frozen fish'], ['other vegetables', 'hygiene articles'], ['whole milk', 'tropical fruit', 'pip fruit'], ['rolls/buns', 'chocolate', 'red/blush wine'], ['other vegetables', 'shopping bags'], ['whole milk', 'rolls/buns', 'chocolate', 'packaged fruit/vegetables'], ['whole milk', 'root vegetables', 'pastry'], ['rolls/buns'], ['whipped/sour cream', 'm

In [ ]:
i = 0
for txn in ordered_transactions:
  i += 1
  print(txn)
  if i==5: break

['whole milk', 'yogurt', 'sausage', 'semi-finished bread']
['whole milk', 'pastry', 'salty snack']
['canned beer', 'misc. beverages']
['sausage', 'hygiene articles']
['soda', 'pickled vegetables']


In [ ]:
# FP tree

class FPNode:
  def __init__(self, item, count, parent=None):
    self.item = item
    self.count = count
    self.parent = parent
    self.children = {}
    self.link = None

In [ ]:
def update_header_table(head, node):
  current = head
  while current.link is not None:
    current = current.link
  current.link = node


def build_FP_tree(ordered_transactions):
  root = FPNode(None,1,None)
  header_table = {}

  for transaction in ordered_transactions:
    current = root

    for item in transaction:
      if item in current.children:
        current.children[item].count += 1

      else:
        new_node = FPNode(item, 1, current)
        current.children[item] = new_node

        if item in header_table:
          update_header_table(header_table[item], new_node)
        else:
          header_table[item] = new_node

      current = current.children[item]

  return root, header_table

In [ ]:
# to get the path by tracing back through parent node
def get_conditional_pattern_base(node):
    patterns = []
    # print(node)
    while node:
        path = []
        parent = node.parent
        while parent and parent.item:
            path.append(parent.item)
            parent = parent.parent
        # print("path:", path)
        if path:
            patterns.append((path[::-1], node.count))
        node = node.link
    return patterns

In [ ]:
def fp_growth(header_table, prefix, freq_itemsets, support_dict):
  for item in header_table:
    new_freq_set = prefix + [item]

    # Support = sum of counts along node links
    count = 0
    node = header_table[item]
    while node:
        count += node.count
        node = node.link

    freq_itemsets.append(new_freq_set)
    support_dict[new_freq_set] = count

    cond_pattern_base = get_conditional_pattern_base(header_table[item])
    # print("==>", cond_pattern_base)
    cond_transactions = []
    for pattern, count in cond_pattern_base:
      for _ in range(count):
        cond_transactions.append(pattern)
    # print(cond_transactions)

    cond_freq = get_item_frequencies(cond_transactions)
    cond_freq = {k: v for k, v in cond_freq.items() if v >= min_support_freq}

    if cond_freq:
        sorted_cond_txns = reorder_transactions(cond_transactions, cond_freq)
        _, cond_header = build_FP_tree(sorted_cond_txns)
        fp_growth(cond_header, new_freq_set, freq_itemsets)

In [ ]:
def fp_growth(header_table, prefix, freq_itemsets, support_dict):
    for item in header_table:
        new_freq_set = tuple(prefix + [item])

        # Support = sum of counts along node links
        count = 0
        node = header_table[item]
        while node:
            count += node.count
            node = node.link

        freq_itemsets.append(new_freq_set)
        support_dict[new_freq_set] = count

        cond_pattern_base = get_conditional_pattern_base(header_table[item])

        cond_transactions = []
        for pattern, cnt in cond_pattern_base:
            for _ in range(cnt):
                cond_transactions.append(pattern)

        cond_freq = get_item_frequencies(cond_transactions)
        cond_freq = {k: v for k, v in cond_freq.items() if v >= min_support_freq}

        if cond_freq:
            sorted_cond_txns = reorder_transactions(cond_transactions, cond_freq)
            _, cond_header = build_FP_tree(sorted_cond_txns)
            fp_growth(cond_header, list(new_freq_set), freq_itemsets, support_dict)


In [ ]:
fp_tree, header_table = build_FP_tree(ordered_transactions)

In [ ]:
frequent_patterns = []
support_dict = {}

fp_growth(header_table, [], frequent_patterns, support_dict)

print("Frequent Itemsets (FP-Growth):")
for pattern in frequent_patterns:
    print(pattern)


Frequent Itemsets (FP-Growth):
('whole milk',)
('yogurt',)
('yogurt', 'whole milk')
('yogurt', 'soda')
('yogurt', 'other vegetables')
('yogurt', 'rolls/buns')
('sausage',)
('sausage', 'whole milk')
('sausage', 'yogurt')
('sausage', 'rolls/buns')
('sausage', 'root vegetables')
('sausage', 'soda')
('sausage', 'other vegetables')
('semi-finished bread',)
('pastry',)
('pastry', 'whole milk')
('pastry', 'other vegetables')
('pastry', 'yogurt')
('pastry', 'soda')
('pastry', 'rolls/buns')
('salty snack',)
('canned beer',)
('canned beer', 'rolls/buns')
('canned beer', 'yogurt')
('canned beer', 'whole milk')
('canned beer', 'other vegetables')
('misc. beverages',)
('hygiene articles',)
('soda',)
('soda', 'whole milk')
('soda', 'other vegetables')
('soda', 'rolls/buns')
('pickled vegetables',)
('frankfurter',)
('frankfurter', 'whole milk')
('frankfurter', 'rolls/buns')
('frankfurter', 'other vegetables')
('curd',)
('curd', 'other vegetables')
('curd', 'whole milk')
('rolls/buns',)
('rolls/buns',

In [ ]:
print(support_dict)

{('whole milk',): 2363, ('yogurt',): 1285, ('yogurt', 'whole milk'): 167, ('yogurt', 'soda'): 87, ('yogurt', 'other vegetables'): 121, ('yogurt', 'rolls/buns'): 117, ('sausage',): 903, ('sausage', 'whole milk'): 134, ('sausage', 'yogurt'): 86, ('sausage', 'rolls/buns'): 80, ('sausage', 'root vegetables'): 50, ('sausage', 'soda'): 89, ('sausage', 'other vegetables'): 90, ('semi-finished bread',): 142, ('pastry',): 774, ('pastry', 'whole milk'): 97, ('pastry', 'other vegetables'): 55, ('pastry', 'yogurt'): 54, ('pastry', 'soda'): 61, ('pastry', 'rolls/buns'): 59, ('salty snack',): 281, ('canned beer',): 702, ('canned beer', 'rolls/buns'): 63, ('canned beer', 'yogurt'): 58, ('canned beer', 'whole milk'): 90, ('canned beer', 'other vegetables'): 60, ('misc. beverages',): 236, ('hygiene articles',): 205, ('soda',): 1453, ('soda', 'whole milk'): 174, ('soda', 'other vegetables'): 145, ('soda', 'rolls/buns'): 121, ('pickled vegetables',): 134, ('frankfurter',): 565, ('frankfurter', 'whole mil

In [ ]:
# get association rules
association_rules = get_association_rules(support_dict)
print("Association rules:")
for rule in association_rules:
   print(rule)
print(f"\nTotal association rules generated: {len(association_rules)}")

Association rules:

Total association rules generated: 0


In [ ]:
from itertools import combinations

minimum_confidence = 0.7

association_rules = []

for itemset, count in support_dict.items():    # here itemset is an array of 1,2,..n-itemset
  if len(itemset) < 2:
    continue
  for i in range(1, len(itemset)):
    for antecedent in combinations(itemset, i):
      consequent = tuple(set(itemset) - set(antecedent))
      if antecedent in frequency and consequent in frequency:
        confidence = get_confidence(antecedent, consequent)
        if confidence >= minimum_confidence:
          association_rules.append(f"[{antecedent} => {consequent}]")

print("Association rules:")
for rule in association_rules: print(rule)

In [ ]:
def print_tree(node, indent=0):
    # Use "Root" for the top-level node (since its item is None)
    item_name = str(node.item) if node.item is not None else "Root"

    # Print the current node's name and its count
    print("  " * indent + f"{item_name}: {node.count}")

    # Recursively print all children
    for child_node in node.children.values():
        print_tree(child_node, indent + 1)

# Usage:
print_tree(fp_tree)

Root: 1
  whole milk: 2363
    yogurt: 116
      sausage: 9
        semi-finished bread: 1
        bottled beer: 1
        shopping bags: 1
          white bread: 1
            sliced cheese: 1
              baking powder: 1
                dog food: 1
        pet care: 1
        pastry: 1
          onions: 1
            sliced cheese: 1
              chewing gum: 1
                finished products: 1
        whipped/sour cream: 1
      tropical fruit: 5
        citrus fruit: 1
          canned beer: 1
            frankfurter: 1
              specialty chocolate: 1
                hard cheese: 1
        bottled water: 1
        pastry: 1
          ham: 1
            hard cheese: 1
              candy: 1
                specialty bar: 1
        canned beer: 1
        newspapers: 1
          dish cleaner: 1
      newspapers: 3
        domestic eggs: 1
        chicken: 1
          soft cheese: 1
            condensed milk: 1
      yogurt: 3
        chewing gum: 1
        yogurt: 1
      